(genai_01_basic_tutorial)=
# Deploying an LLM using MLRun
This notebook illustrates deploying an LLM using MLRun: it shows how to grab a dataset, which is a list of articles, scrape those articles, chunk and index them into a vector store, and then deploy an open-source Hugging Face model to an endpoint where you can make requests and get responses with a RAG enrichment using the data that you just downloaded.

Since this tutorial is for illustrative purposes, it uses minimal resources &mdash; CPU and not GPU, and a small amount of data.

**In this tutorial:**
- [MLRun installation and configuration](#mlrun-installation-and-configuration)
- [Set up the vector database in the cluster](#set-up-the-vector-database-in-the-cluster)
- [Build the vector DB](#build-the-vector-db)
- [Serving the function](#serving-the-function)

**See also:**
- [Model monitoring](https://docs.mlrun.org/en/stable/concepts/model-monitoring.html)
- [Alerts-notifications](https://docs.mlrun.org/en/stable/concepts/alerts-notifications.html)

<iframe width="560" height="315" src="https://www.youtube.com/embed/aoP__SaAO1M" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" allowfullscreen></iframe><br><br>

## MLRun installation and configuration

Before running this notebook make sure the `mlrun` packages are installed (`pip install mlrun`) and that you have configured the access to MLRun service. 

In [1]:
# Install MLRun if not installed, run this only once. Restart the notebook after the install
# %pip install mlrun

In [1]:
import json
import mlrun

**Get or create a new project**

First create, load or use (get) an [MLRun Project](https://docs.mlrun.org/en/stable/projects/project.html). The [get_or_create_project](https://docs.mlrun.org/en/stable/api/mlrun.projects/index.html#mlrun.projects.get_or_create_project) method tries to load the project from the MLRun DB. If the project does not exist, it creates a new one.

In [4]:
project = mlrun.get_or_create_project(
    "genai-tutorial", "./", user_project=True, allow_cross_project=True
)
project.set_source(".", pull_at_runtime=True)

> 2025-09-12 04:27:00,942 [info] Project loaded successfully: {"project_name":"genai-tutorial-xingsheng"}


## Set up the vector database in the cluster 
These two steps imports a pre-defined dataset and load it into a vector database. Then the vector database is stored in the data layer of the cluster.

If you're not using Iguazio's Jupyter, download {download}`fetch-vectordb-data.py <src/fetch-vectordb-data.py>`.

In [4]:
# The model used is the free open-source PHI 2
MODEL_ID = "microsoft/phi-2"

# Define the dataset for the VectorDB
DATA_SET = mlrun.get_sample_path("data/genai-tutorial/labelled_newscatcher_dataset.csv")

# The location of the VectorDB files
CACHE_DIR = mlrun.mlconf.artifact_path
CACHE_DIR = (
    CACHE_DIR.replace("v3io://", "/v3io").replace("{{run.project}}", project.name)
    + "/cache"
)

In [5]:
is_ce = CACHE_DIR.startswith("s3://")
is_ce

True

### build an image from mlrun/mlrun to include langchain and torch packages

The image can be created with mlrun/mlrun as base:
```
'pip install chromadb==0.5.0 langchain==0.2.3 langchain-community==0.2.4 langchain-core==0.2.5 langchain-text-splitters==0.2.1 clean-text==0.6.0 transformers==4.41.2',
'pip install torch --index-url https://download.pytorch.org/whl/cpu',
'pip install protobuf==3.20.2',
'pip install requests-toolbelt==0.10.1',
```

In [6]:
commands = [
    "pip install chromadb==0.5.0 langchain==0.2.3 langchain-community==0.2.4 langchain-core==0.2.5 langchain-text-splitters==0.2.1 clean-text==0.6.0 transformers==4.41.2",
    "pip install torch --index-url https://download.pytorch.org/whl/cpu",
    "pip install requests-toolbelt==0.10.1",
]

In [7]:
# need different version of protobuf to work with different python version, python 3.9 -> protobuf 3.20.1, python 3.11 -> protobuf latest
import sys

minor_version = float(sys.version_info[1])
if minor_version >= 11:
    commands.append("pip install --upgrade --force-reinstall protobuf")
elif minor_version == 9:
    commands.append("pip install protobuf==3.20.2")
else:
    print(f"minor_version {minor_version} not supported")
commands

['pip install chromadb==0.5.0 langchain==0.2.3 langchain-community==0.2.4 langchain-core==0.2.5 langchain-text-splitters==0.2.1 clean-text==0.6.0 transformers==4.41.2',
 'pip install torch --index-url https://download.pytorch.org/whl/cpu',
 'pip install requests-toolbelt==0.10.1',
 'pip install protobuf==3.20.2']

Run the following command to build the image, once it's successfully built, no need to run the cell again since it takes quite sometime to build an image.

In [ ]:
project.build_image(
    image=".llm-demo-data",
    base_image="mlrun/mlrun",
    set_as_default=False,
    commands=commands,
)

Fetch the dataset for the Vector DB and save it in cluster:

In [9]:
fetch = project.set_function(
    name="fetch-vectordb-data",
    func="src/fetch-vectordb-data.py",
    kind="job",
    image=".llm-demo-data",
)

In [10]:
ret = project.run_function(
    name="fetch-vectordb-data-run",
    function="fetch-vectordb-data",
    handler="handler",
    params={"data_set": DATA_SET},
)

> 2025-08-08 03:44:08,003 [info] Storing function: {"db":"http://mlrun-api:8080","name":"fetch-vectordb-data-run","uid":"8fb490a756d94618870a8c4662d4afd4"}
> 2025-08-08 03:44:08,275 [info] Job is running in the background, pod: fetch-vectordb-data-run-2sfpd
Since the GPL-licensed package `unidecode` is not installed, using Python's `unicodedata` package which yields worse results.
Fetching pages: 100%|##########| 80/80 [00:40<00:00,  1.98it/s]
> 2025-08-08 03:45:28,304 [info] Dataset dowloaded and logged
> 2025-08-08 03:45:28,402 [info] To track results use the CLI: {"info_cmd":"mlrun get run 8fb490a756d94618870a8c4662d4afd4 -p genai-tutorial-jovyan","logs_cmd":"mlrun logs 8fb490a756d94618870a8c4662d4afd4 -p genai-tutorial-jovyan"}
> 2025-08-08 03:45:28,403 [info] Run execution finished: {"name":"fetch-vectordb-data-run","status":"completed"}


project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results,artifacts
genai-tutorial-jovyan,...62d4afd4,0,Aug 08 03:44:31,2025-08-08 03:45:28.385057+00:00,completed,run,fetch-vectordb-data-run,kind=jobowner=jovyanmlrun/client_version=1.9.2mlrun/client_python_version=3.9.13host=fetch-vectordb-data-run-2sfpd,,data_set=https://s3.wasabisys.com/iguazio/data/genai-tutorial/labelled_newscatcher_dataset.csv,,vector-db-dataset


> 2025-08-08 03:45:33,773 [info] Run execution finished: {"name":"fetch-vectordb-data-run","status":"completed"}


In [11]:
ret.outputs

{'vector-db-dataset': 'store://datasets/genai-tutorial-jovyan/fetch-vectordb-data-run_vector-db-dataset:latest@8fb490a756d94618870a8c4662d4afd4^2a332da88f7faa2b7d044b2818fec436f98133f2'}

## Build the vector DB

Build the vector DB in the data layer and load the data into it.


If you're not using Iguazio's Jupyter, download {download}`the build vector db <./src/build-vector-db.py>`.

In [12]:
# Build the vector DB using the image
build_vectordb = project.set_function(
    name="build-vectordb",
    func="src/build-vector-db.py",
    kind="job",
    image=".llm-demo-data",
)

In [13]:
if not is_ce:
    build_vectordb.apply(mlrun.auto_mount())
    print("Applying mlrun.auto_mount!")
else:
    print("Not applying mlrun.auto_mount!")

Not applying mlrun.auto_mount!


In [ ]:
build_vectordb_run = project.run_function(
    function="build-vectordb",
    inputs={"df": ret.outputs["vector-db-dataset"]},
    params={"cache_dir": CACHE_DIR},
    handler="handler_chroma",
    outputs=["vect_db"],
)

In [15]:
VECTORDB_PATH = build_vectordb_run.outputs["vect_db"]

## Serving the function

If you're not using Iguazio's Jupyter, download {download}`serving.py <./src/serving.py>`.
Now you can deploy the the Nuclio function that serves the LLM:

In [16]:
serve_func = project.set_function(
    name="serve-llm",
    func="src/serving.py",
    image=".llm-demo-data",
    kind="nuclio",
)

# Transferring the model and VectorDB path to the serving functions
serve_func.set_envs(
    env_vars={
        "MODEL_ID": MODEL_ID,
        "CACHE_DIR": CACHE_DIR,
        "VECTORDB_PATH": VECTORDB_PATH,
    }
)

# Since the model is stored in memory, use only 1 replica and and one worker
# Since this is running on CPU only, inference might take ~1 minute (increasing timeout)
serve_func.spec.min_replicas = 1
serve_func.spec.max_replicas = 1
serve_func.with_http(worker_timeout=120, gateway_timeout=150, workers=1)
serve_func.set_config("spec.readinessTimeoutSeconds", 1200)

> 2025-08-08 04:05:21,460 [warning] Adding HTTP trigger despite the default HTTP trigger creation being disabled


In [17]:
if not is_ce:
    serve_func.apply(mlrun.auto_mount())
    print("Applying mlrun.auto_mount!")
else:
    print("Not applying mlrun.auto_mount!")

Not applying mlrun.auto_mount!


In [18]:
serve_func = project.deploy_function(function="serve-llm")

> 2025-08-08 04:05:21,482 [info] Starting remote function deploy
2025-08-08 04:05:21  (info) Deploying function
2025-08-08 04:05:21  (info) Building
2025-08-08 04:05:21  (info) Staging files and preparing base images
2025-08-08 04:05:21  (warn) Using user provided base image, runtime interpreter version is provided by the base image
2025-08-08 04:05:21  (info) Building processor image
2025-08-08 04:08:12  (info) Build complete
2025-08-08 04:09:24  (info) Function deploy complete
> 2025-08-08 04:09:27,286 [info] Successfully deployed function: {"external_invocation_urls":["genai-tutorial-jovyan-serve-llm-xingsheng-large-l.genai.mckinsey.cloud/"],"internal_invocation_urls":["nuclio-genai-tutorial-jovyan-serve-llm.xingsheng-large-l.svc.cluster.local:8080"]}


### Test Serving Function
The inference endpoint of a LLM which is hosted in the cluster

In [19]:
body = {
    "question": "What are some new developments in space travel?",
    "topic": "science",
}

In [20]:
resp = serve_func.function.invoke("/", body=json.dumps(body))

In [21]:
print(resp["response"])


Some new developments in space travel include the successful launch of a rocket from Earth, which poses a threat to space travel. Additionally, there have been advancements in the field of science, with new discoveries and breakthroughs being made. The news also covers topics such as politics, royal affairs, showbiz and TV, sports, finance, travel, life and style, comment, and world news.



In [22]:
print(resp["sources"])

['https://www.express.co.uk/news/science/1324095/space-news-spacex-rocket-launch-stars-elon-musk-Comet-Neowise-latest']


In [23]:
print(resp["prompt"])

The instruction below describes a task. Write a response that appropriately completes the request.

### Instruction:
User question:
What are some new developments in space travel?

Context:
Space news: Rocket launches from earth pose threat to space travel | Science | News | Express.co.uk Express. Home of the Daily and Sunday Express. Puzzles Horoscopes Express Rated Shop Paper Newsletters Login Register Your Account Newsletters Bookmarks Sign OutUkUs 19C Find us on FacebookFollow us on WhatsApp Follow us on TwitterFind us on Instagram Find us on Youtube Search HOME News Politics Royal Showbiz & TV Sport Finance Travel Life & Style Comment UK World Politics Royal US Weather Science

### Response:



In [24]:
project.set_function(f"db://{project.name}/fetch-vectordb-data")
project.set_function(f"db://{project.name}/build-vectordb")
project.set_function(f"db://{project.name}/serve-llm")
project.set_source(f"db://{project.name}")
project.save()

### Run E2E Workflow

In [25]:
%%writefile workflow.py
import mlrun
from kfp import dsl

    
@dsl.pipeline(
    name="GenAI demo"
)

def kfpipeline(data_set, cache_dir, model_id):
    
    project = mlrun.get_current_project()
    
    fetch = project.run_function(
        function="fetch-vectordb-data",
        name="fetch-vectordb-data-run",
        handler="handler",
        params = {"data_set" : data_set},
        outputs=['vector-db-dataset']
    )
    
    
    vectordb_build = project.run_function(
        function="build-vectordb",
        inputs={"df" : fetch.outputs["vector-db-dataset"]},
        params={"cache_dir" : cache_dir},
        handler="handler_chroma",
        outputs=["vect_db"]        
    )

    serve_func = project.get_function("serve-llm")
    serve_func.set_envs(
        env_vars={"MODEL_ID": model_id, 
                  "CACHE_DIR": cache_dir, 
                  "VECTORDB_PATH":vectordb_build.outputs["vect_db"]}
    )
    serve_func.spec.min_replicas = 1
    serve_func.spec.max_replicas = 1
    serve_func.with_http(worker_timeout=120, gateway_timeout=150, workers=1)
    serve_func.set_config("spec.readinessTimeoutSeconds", 1200)
    
    deploy = project.deploy_function("serve-llm", verbose=True).after(vectordb_build)

Overwriting workflow.py


In [26]:
project.set_workflow("main", "workflow.py", embed=True)
project.save()

#### Please note that the workflow may take up to 20 mins to complete.

In [ ]:
run_id = project.run(
    "main",
    arguments={"cache_dir": CACHE_DIR, "data_set": DATA_SET, "model_id": MODEL_ID},
    engine="remote",
    watch=True,
)